In [1]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
import sys
sys.path.append("..")

# ── Load data ────────────────────────────────────────────────────────
with open("../data/puzzles_labeled.json", "r") as f:
    puzzles = json.load(f)

# fix words and remove emoji puzzle
for puzzle in puzzles:
    words_from_groups = []
    for group in puzzle["groups"]:
        words_from_groups.extend(group["members"])
    puzzle["words"] = words_from_groups
puzzles = [p for p in puzzles if p["puzzle_id"] != 295]

# ── Train/test split by date ─────────────────────────────────────────
puzzles_sorted = sorted(puzzles, key=lambda p: p["date"])
split_idx = int(len(puzzles_sorted) * 0.80)
train_puzzles = puzzles_sorted[:split_idx]
test_puzzles = puzzles_sorted[split_idx:]

print(f"Train: {len(train_puzzles)} puzzles")
print(f"Test:  {len(test_puzzles)} puzzles")
print(f"Train date range: {train_puzzles[0]['date']} to {train_puzzles[-1]['date']}")
print(f"Test date range:  {test_puzzles[0]['date']} to {test_puzzles[-1]['date']}")

Train: 180 puzzles
Test:  46 puzzles
Train date range: 2023-10-16 to 2024-04-18
Test date range:  2024-04-19 to 2024-06-03


In [2]:
# ── Load MPNet and compute embeddings ────────────────────────────────
print("Loading MPNet...")
model = SentenceTransformer("all-mpnet-base-v2")
print("Model loaded")

print("Computing embeddings...")
for puzzle in puzzles_sorted:
    embeddings = model.encode(puzzle["words"], show_progress_bar=False)
    puzzle["embeddings"] = embeddings

print(f"Embeddings computed for {len(puzzles_sorted)} puzzles")

Loading MPNet...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Computing embeddings...
Embeddings computed for 226 puzzles


In [3]:
from solver.candidates import generate_candidates
from solver.scoring import score_all_candidates
from solver.search import greedy_solve, beam_solve, get_tau_estimate
from solver.evaluate import run_evaluation

print("Solver modules imported successfully")

Solver modules imported successfully


In [4]:
# ── Full ablation evaluation on test set ─────────────────────────────
import copy
from solver.candidates import generate_candidates
from solver.scoring import score_all_candidates
from solver.search import greedy_solve, beam_solve, get_tau_estimate
from solver.lexical import add_lexical_scores
from solver.evaluate import run_evaluation, evaluate_prediction
from solver.feedback import apply_feedback, simulate_feedback

# best weights from grid search
ALPHA = 1.0
BETA  = 0.0
GAMMA = 0.05
ETA   = 0.3

# precompute candidates for test puzzles once
print("Precomputing test candidates...")
test_precomputed = []
for puzzle in test_puzzles:
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = generate_candidates(words, embeddings)
    tau = get_tau_estimate(candidates)
    test_precomputed.append({
        "puzzle": puzzle,
        "candidates": candidates,
        "tau": tau
    })
print(f"Done — {len(test_precomputed)} puzzles precomputed")

Precomputing test candidates...
Done — 46 puzzles precomputed


In [9]:
# ── Solver 1: Greedy ─────────────────────────────────────────────────
solved_s1, correct_s1, top1_s1 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    predicted = greedy_solve(words, embeddings, candidates)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s1.append(result["solved"])
    correct_s1.append(result["n_correct"])
    top1_s1.append(result["top1_correct"])

# ── Solver 2: Beam only ──────────────────────────────────────────────
solved_s2, correct_s2, top1_s2 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s2.append(result["solved"])
    correct_s2.append(result["n_correct"])
    top1_s2.append(result["top1_correct"])

# ── Solver 3: Beam + penalty ─────────────────────────────────────────
solved_s3, correct_s3, top1_s3 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s3.append(result["solved"])
    correct_s3.append(result["n_correct"])
    top1_s3.append(result["top1_correct"])

print("Solvers 1-3 done")

Solvers 1-3 done


In [5]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/benjaminburda/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/benjaminburda/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [10]:
# ── Solver 4: Beam + penalty + lexical ───────────────────────────────
solved_s4, correct_s4, top1_s4 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    candidates = add_lexical_scores(candidates)
    for c in candidates:
        c["score"] = c["score"] + ETA * c["lexical"]
    candidates.sort(key=lambda x: x["score"], reverse=True)
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s4.append(result["solved"])
    correct_s4.append(result["n_correct"])
    top1_s4.append(result["top1_correct"])

print("Solver 4 done")

Solver 4 done


In [8]:
# ── Solver 5 final ───────────────────────────────────────────────────
solved_s5, correct_s5, top1_s5 = [], [], []

for item in test_precomputed:
    puzzle = item["puzzle"]
    true_groups = puzzle["groups"]
    remaining = list(puzzle["words"])
    solved_groups = []
    excluded_guesses = set()

    for _ in range(7):
        if len(solved_groups) == 4:
            break
        if len(remaining) == 4:
            solved_groups.append(remaining)
            break

        remaining_embs = model.encode(remaining, show_progress_bar=False)
        candidates = generate_candidates(remaining, remaining_embs)
        candidates = [c for c in candidates if c["word_set"] not in excluded_guesses]
        if not candidates:
            break

        tau_current = get_tau_estimate(candidates)
        candidates = score_all_candidates(candidates, tau_current, ALPHA, BETA, GAMMA)
        candidates = add_lexical_scores(candidates)
        for c in candidates:
            c["score"] = c["score"] + ETA * c["lexical"]
        candidates.sort(key=lambda x: x["score"], reverse=True)

        try:
            predicted = beam_solve(remaining, remaining_embs, candidates, beam_width=25)
            if not predicted:
                break
            guess = predicted[0]
        except Exception:
            break

        feedback = simulate_feedback(guess, true_groups)
        if feedback == "correct":
            solved_groups.append(guess)
            remaining = [w for w in remaining if w not in guess]
            excluded_guesses = set()
        else:
            excluded_guesses.add(frozenset(guess))

    result = evaluate_prediction(solved_groups, true_groups)
    solved_s5.append(result["solved"])
    correct_s5.append(result["n_correct"])
    top1_s5.append(result["top1_correct"])

print("Solver 5 final done")
print(f"Solve rate:          {np.mean(solved_s5):.3f}")
print(f"Mean correct groups: {np.mean(correct_s5):.3f}")
print(f"Top-1 accuracy:      {np.mean(top1_s5):.3f}")

Solver 5 final done
Solve rate:          0.174
Mean correct groups: 1.261
Top-1 accuracy:      0.587


In [11]:
# ── Updated full ablation results table ──────────────────────────────
results_summary = pd.DataFrame([
    {
        "Solver": "S1: Greedy",
        "Solve Rate": np.mean(solved_s1),
        "Mean Correct Groups": np.mean(correct_s1),
        "Top-1 Accuracy": np.mean(top1_s1)
    },
    {
        "Solver": "S2: Beam",
        "Solve Rate": np.mean(solved_s2),
        "Mean Correct Groups": np.mean(correct_s2),
        "Top-1 Accuracy": np.mean(top1_s2)
    },
    {
        "Solver": "S3: Beam + Penalty",
        "Solve Rate": np.mean(solved_s3),
        "Mean Correct Groups": np.mean(correct_s3),
        "Top-1 Accuracy": np.mean(top1_s3)
    },
    {
        "Solver": "S4: Beam + Penalty + Lexical",
        "Solve Rate": np.mean(solved_s4),
        "Mean Correct Groups": np.mean(correct_s4),
        "Top-1 Accuracy": np.mean(top1_s4)
    },
    {
        "Solver": "S5: Full Pipeline + Feedback",
        "Solve Rate": np.mean(solved_s5),
        "Mean Correct Groups": np.mean(correct_s5),
        "Top-1 Accuracy": np.mean(top1_s5)
    }
])

print(results_summary.round(3).to_string(index=False))

                      Solver  Solve Rate  Mean Correct Groups  Top-1 Accuracy
                  S1: Greedy       0.000                0.457           0.217
                    S2: Beam       0.065                0.717           0.261
          S3: Beam + Penalty       0.065                0.717           0.261
S4: Beam + Penalty + Lexical       0.043                0.739           0.348
S5: Full Pipeline + Feedback       0.174                1.261           0.587


In [8]:
from dotenv import load_dotenv
import os
load_dotenv("../.env")
client = anthropic.Anthropic()

In [9]:
from solver.categorize import classify_group, classify_all_puzzles
import anthropic

client = anthropic.Anthropic()

# test on a few groups
test_groups = [
    ("DAYS OF THE WEEK", ["FRIDAY", "SATURDAY", "SUNDAY", "THURSDAY"]),
    ("___LAND", ["FIN", "GREEN", "ICE", "IRE"]),
    ("W.N.B.A. TEAMS", ["LIBERTY", "MERCURY", "SKY", "SPARKS"]),
    ("THINGS WITH TRUNKS", ["CARS", "ELEPHANTS", "SWIMMERS", "TREES"]),
    ("SHAMELESS BOLDNESS", ["BRASS", "CHEEK", "GALL", "NERVE"]),
]

for description, members in test_groups:
    result = classify_group(description, members, client)
    print(f"{description}: {result['category']} — {result['reasoning']}")

DAYS OF THE WEEK: SEMANTIC — This group consists of days of the week, which form a clear semantic category based on a direct conceptual relationship - they are all members of the same semantic class.
___LAND: WORDPLAY — This group consists of words that can all be combined with 'LAND' to form compound words or place names: FINLAND, GREENLAND, ICELAND, IRELAND. This is a classic wordplay pattern based on word formation.
W.N.B.A. TEAMS: ENCYCLOPEDIC — This group requires knowledge of professional women's basketball teams in the WNBA, which is cultural/sports knowledge rather than semantic relationships between the words themselves.
THINGS WITH TRUNKS: ASSOCIATIVE — This group connects items that all have 'trunks' but in different senses - car trunks (storage compartment), elephant trunks (body part), swimmer trunks (clothing), and tree trunks (main stem). This is a thematic connection based on shared terminology rather than semantic similarity or wordplay.
SHAMELESS BOLDNESS: SEMANTIC — 

In [5]:
# ── Classify all groups in labeled puzzles ───────────────────────────
from solver.categorize import classify_all_puzzles
import json

print("Classifying all groups...")
classifications = classify_all_puzzles(puzzles_sorted, delay=0.1)

# save to disk so we don't have to rerun
with open("../data/group_classifications.json", "w") as f:
    json.dump(classifications, f, indent=2)

print(f"Done — {len(classifications)} groups classified")

# quick summary
from collections import Counter
cats = Counter(v["category"] for v in classifications.values())
print("\nCategory distribution:")
for cat, count in cats.most_common():
    print(f"  {cat}: {count}")

Classifying all groups...
  50/904 groups classified


RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 50 requests per minute (org: be3cace4-6c5d-4136-9f5f-151af7800846, model: claude-sonnet-4-20250514). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CZfCSp5dGhjnFhJiqg95z'}